# Project Web dataming "The history and culture of Lebanon" : Eva BERRY and Charles AYOUB TD DIA 1

# Introduction

In [ ]:
import requests

url = "https://upload.wikimedia.org/wikipedia/commons/thumb/5/59/Flag_of_Lebanon.svg/1280px-Flag_of_Lebanon.svg.png"

In [ ]:
from IPython.display import display, HTML
display(HTML(f"""
<img src="{url}" style="
    display: block;
    margin: auto;
    width: 300px;
    border-radius: 10px;
">
"""))

Nowadays, the world keeps evolving and become bigger day by day. It’s important to put interest in other countries’ habits and try to discover as much as possible new cultures. In fact, it’s not only important for our personal culture but also for our professional life, as we can be lead to collaborate with international people. 
As part of the web data mining project, we wanted to take this opportunity to share our attachment towards our originary and lovely country: Lebanon, the Swiss of the middle east.  

In this study, we will have the opportunity to discover the history of Lebanon, its culture, the places and most famous cities to visit as well as the friendship between France and Lebanon. 

So, in order to conduct this study, we have chosen few websites that displays those informations and we are going to use web scrapping to explore them, as well as the SpaCy and NER tools in order to filter those pages. Finally, we will use the RDF graph in order to feed our chatbot based on an ollama model. 


# Phase 1 : Web Crawling and Cleaning

## 1.1 Source Selection

In [ ]:
import json
import requests
import pandas as pd
import spacy
import trafilatura as tf
from bs4 import BeautifulSoup

In [ ]:
max_pages = 10
visited = set()

queue = [
    "https://en.wikipedia.org/wiki/Histoire_du_Liban",
    "https://en.wikipedia.org/wiki/Beyrouth",
    "https://en.wikipedia.org/wiki/Tripoli,_Lebanon",
    "https://en.wikipedia.org/wiki/Jezzine",
    "https://en.wikipedia.org/wiki/France%E2%80%93Lebanon_relations",
    "https://www.artshelp.com/mim-museum/"
]

## 1.2 The Extraction Pipeline

**Function for checking that the page have at least 500 words:**

We initialize a list nammed "records" that will contain all the valide pages with their URL, title and text content.
Then, we created a function nammed "has_min_words" that allows us to keep only the pages that have at least 500 words. Basically, we split the block texts into words and we count the words' number. If we have a minimum of 500 words in the page, we keep it.

In [ ]:
records = []

def has_min_words(text, min_words=500):
    if not text:
        return False
    return len(text.split()) >= min_words

**Pipeline that extract the page, its core data and the check of 500 words minimum recquired:**

Then, we created a "while" loop that allows us to iterate on all the URLs. For each URL, we avoid the duplicates and then we download the HTML page by making an HTPP request. 

In addition, we use the trafilatura function to extract the relevant text of the page. In that case, we will be able to put aside the ads and the menus.

Then, we stock the extracted text into a JSON file 'data' using the function "json.loads(main_text)".

Then, we extract the title using a "get" request from the "data" json. However, if this action fails, we use the "BeautifulSoup" library in order to transform the HTML into a readable tree so that we can then extract the title.

In [ ]:
while queue and len(visited) < max_pages:

    url = queue.pop(0)

    if url in visited:
        continue

    visited.add(url)
    print(f"\nVisiting: {url}")

    try:
        headers = {
            "User-Agent": "Mozilla/5.0 (educational crawler)"
        }

        response = requests.get(url, headers=headers, timeout=5)
        response.raise_for_status()

        html = response.text

    except Exception as e:
        print("Failed:", e)
        continue


    main_text = tf.extract(
        html,
        output_format="json",
        include_comments=False,
        include_tables=False,
        no_fallback=True
    )

    if main_text is None:
        print("Extraction failed")
        continue


    data = json.loads(main_text)

    title = data.get("title", "unknown")

    if title == "unknown":
        soup = BeautifulSoup(html, "html.parser")
        if soup.title:
            title = soup.title.get_text()

    text = data.get("text", "")

    if not has_min_words(text, 500):
        print("Too short")
        continue


    print("Title:", title)
    print("Words:", len(text.split()))

    records.append({
        "url": url,
        "title": title,
        "text": text
    })


print("Pages kept:", len(records))

**Stock of the result in a JSONL format:**

Finally, we stock the results into a JSONL formats. In each line of this JSONL document, we will have the URL of the website, its title and its text content.

In [ ]:
filename = "crawler_output.jsonl"

with open(filename, "w", encoding="utf-8") as f:

    for record in records:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")


print("Saved crawler_output.jsonl")

# Phase 2 : Information Extraction

## 2.1 Named Entity Recognition (NER)

**Load of the NLP pipeline in order to filter the text in the english language:**

!python -m spacy download en_core_web_trf

In [ ]:
nlp = spacy.load("en_core_web_trf")

Now, let's initialize the "rows" list that will contain all our entities. 

Then, we declare the common words that we found in our pages and we class them in a dictionnary. The goal is to ignore the words that are not specifically an entity.

Then, we use our previous jsonl file "crawler_output" and each line is tranformed into a dictionnary. Then, we use Spacy (NLP model) in order to analyze the extracted text and for each entity we assign to it a category that match between : ["PERSON", "ORG", "GPE", "LOC", "DATE"].

Finally, we clean the texts and we delete the spaces and the small words such as "the" "so" ect. Moreover, we will keep only the entities that have an uppercase at the begining of the entity because usually the entities takes that form. In addition, we will keep only one word if we have duplicates.

In [ ]:
rows = []
seen = set()

with open("crawler_output.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        obj = json.loads(line)

        url = obj["url"]
        title = obj.get("title", "")
        text = obj.get("text", "")

        if not text:
            continue

        doc = nlp(text)

        for ent in doc.ents:
            if ent.label_ not in ["PERSON", "ORG", "GPE", "LOC", "DATE"]:
                continue

            entity_text = ent.text.strip()

            if not entity_text:
                continue

            if len(entity_text) < 3:
                continue

            entity_text_lower = entity_text.lower()

            key = (entity_text_lower, ent.label_, url)
            if key in seen:
                continue

            seen.add(key)

            rows.append({
                "entity": entity_text,
                "label": ent.label_,
                "source_url": url,
                "title": title
            })

print("Entities extracted:", len(rows))

Finnally, we stocked the entities and their categories into a csv file nammed "extracted_knowledge".

In [ ]:
df = pd.DataFrame(rows)

df = df[df["entity"].str.len() > 2]

df = df.drop_duplicates(
    subset=["entity", "label", "source_url"]
)

df.to_csv(
    "extracted_knowledge.csv",
    index=False,
    encoding="utf-8"
)

print("Saved extracted_knowledge.csv")
print("Number of entities:", len(df))

df.head(20)

## 2.2 Introduction to Relations

Now, we will see the relations between the entities. In fact, for each entity, we will try to extract the subject-verb-predicate. 

First, we will use the "crawler_output" jsonl file and we will extract the verb. Then, based on the verb we will try to catch the subject and the object.

Finally, we will store everything in a csv file nammed "relations.csv".

In [ ]:
relations = []

with open("crawler_output.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        obj = json.loads(line)

        url = obj["url"]
        text = obj["text"]

        doc = nlp(text)

        for sent in doc.sents:
            ents_in_sent = list(sent.ents)
            if len(ents_in_sent) < 2:
                continue

            for tok in sent:
                if tok.pos_ != "VERB":
                    continue

                subj = None
                obj_token = None

                for child in tok.children:
                    if child.dep_ in ("nsubj", "nsubjpass"):
                        subj = child
                    if child.dep_ in ("dobj", "pobj", "attr", "iobj", "obj"):
                        obj_token = child

                if subj is not None and obj_token is not None:
                    relations.append({
                        "subject": subj.text,
                        "predicate": tok.lemma_,
                        "object": obj_token.text,
                        "sentence": sent.text
                    })

rel_df = pd.DataFrame(relations).drop_duplicates()
print("Relations:", len(rel_df))
rel_df.to_csv("relations.csv", index=False)

# Phase 3 : Build the Initial Private Knowledge Base

Now that we have our relations, let's generate the RDF graph. In order to do that, we will translate the relations by creating URIs for each entity and its respective subject, predicate and object. Finally, all the URIs will be stocked in the ".nt" file.

In [ ]:
from rdflib import Graph, URIRef

def clean_name_for_uri(text):
    clean = (
        str(text)
        .strip()
        .replace("\n", " ")
        .replace("\r", " ")
        .replace("\t", " ")
        .replace(" ", "_")
        .replace("/", "_")
        .replace(",", "")
        .replace("(", "")
        .replace(")", "")
        .replace("'", "")
        .replace('"', "")
        .replace("[", "")
        .replace("]", "")
        .replace("{", "")
        .replace("}", "")
        .replace("<", "")
        .replace(">", "")
        .replace("?", "")
        .replace("#", "")
        .replace("&", "")
        .replace(";", "")
        .replace(":", "")
    )
    return clean

g = Graph()

def to_uri(text):
    clean = clean_name_for_uri(text)
    return "http://example.org/entity/" + clean

def to_predicate(text):
    clean = clean_name_for_uri(text)
    return "http://example.org/relation/" + clean

for r in relations:
    subject = str(r["subject"]).strip()
    predicate = str(r["predicate"]).strip()
    obj = str(r["object"]).strip()

    if not subject or not predicate or not obj:
        continue

    s = URIRef(to_uri(subject))
    p = URIRef(to_predicate(predicate))
    o = URIRef(to_uri(obj))

    g.add((s, p, o))

g.serialize("knowledge_base.nt", format="nt", encoding="utf-8")

# Phase 4 : Entity Linking with Open Knowledge Bases

In [ ]:
import pandas as pd
import requests
import time
from difflib import SequenceMatcher
from rdflib import Graph, URIRef, Literal, Namespace, RDF, RDFS

**Configuration:**

Here, we will fix all the links that we will need for accessing wikidata API as well as the base of the local entities and relations. In addition, we will also add the variables for dowloding the files of mapping, entity alignement, entity linking mapping, the entities not found and finally the enriched graph.

In [ ]:
WIKIDATA_API = "https://www.wikidata.org/w/api.php"
USER_AGENT = "KBProject/1.0 (student project)"
OWL_SAMEAS = URIRef("http://www.w3.org/2002/07/owl#sameAs")

LOCAL_ENTITY_BASE = "http://example.org/entity/"
LOCAL_RELATION_BASE = "http://example.org/relation/"

EXTRACTED_ENTITIES_FILE = "extracted_knowledge.csv"
KNOWLEDGE_BASE_INPUT = "knowledge_base.nt"

MAPPING_OUTPUT = "wikidata_mapping.csv"
ALIGNMENT_OUTPUT = "entity_alignment.nt"
FOUND_OUTPUT = "entity_linking_mapping.csv"
NOT_FOUND_OUTPUT = "entity_linking_not_found.csv"
ENRICHED_KB_OUTPUT = "knowledge_base_enriched.nt"

**Normalization of the text:**

Before searching the equivalent of the entities in wikidata, we need to normalize the text and therefore we will use the "normalize_text" function in order to clean the texts and make all the texts in lower cases.

In [ ]:
def normalize_text(text):
    return str(text).strip().lower().replace("_", " ")

**Calculation of the similarity:**

Then, for each entities we will need to calculate the confidence rate. In order to do that, we will use the "SequenceMatcher" library in order to compare words by words the two under-commun sequences. Moreover, the function ".ratio()" allows us to return a score of similarity between "0" and "1". The computation of this ratio take this form:

similarity = (2*M)/T

with M = number of common characters
and  T = total length of the two characters sequences

In [ ]:
def similarity(a, b):
    return SequenceMatcher(None, normalize_text(a), normalize_text(b)).ratio()

**Creation of clean URI:**

Once we have compared the words and define the similarity computation, we need to clean the name of the URIs and generate clean URIs using the functions "clean_name_for_uri" and "make_local_uri". 


In [ ]:
def clean_name_for_uri(text):
    clean = (
        str(text)
        .strip()
        .replace("\n", " ")
        .replace("\r", " ")
        .replace("\t", " ")
        .replace(" ", "_")
        .replace("/", "_")
        .replace(",", "")
        .replace(".", "_")
        .replace("(", "")
        .replace(")", "")
        .replace("'", "")
        .replace('"', "")
        .replace("[", "")
        .replace("]", "")
        .replace("{", "")
        .replace("}", "")
        .replace("<", "")
        .replace(">", "")
        .replace("?", "")
        .replace("#", "")
        .replace("&", "")
        .replace(";", "")
        .replace(":", "")
    )
    return clean

def make_local_uri(name):
    clean = clean_name_for_uri(name)
    return f"{LOCAL_ENTITY_BASE}{clean}"

Once we have done this cleaning and the similarity computation, we can now search and extract the informations from Wikidata. In order to do that, we will use the WIKIDATA API. The function take in parameter the entity and then for each entity, we will search it's equivalent in wikidata. In fact, we prepare an HTPP request for the API by giving the entity search endpoint, the entity name, the language (english as our websites and entities are in english) and the limit of number of results. Then, we send the request by using a GET and we recuperate the JSON file with the answer.

Concerning the result obtained from WIKIDATA API, if there is no match in wikidata, it doesn't return an URI and therefore the confidence is equal to zero. For the ones that have a match on WIKIDATA, we will extract its informations and compute the similarity score so that we can have the its confidence score. Moreover, for each entity that have a equivalent on wikidata we try to recuperate its label and description and in the case where we have a description, we will add a 0.05 on its confidence score because it's a trustfull match. Indeed, for each entiry, we usually have few candidates on WIKIDATA, so that's why we make a loop on those candidates each time and we select the one that have the best confidence score.

Finally, we create the URI and we stock them in the mapping table in the next parts.

In [ ]:
def search_wikidata(entity_name, lang="en", limit=5):
    params = {
        "action": "wbsearchentities",
        "format": "json",
        "language": lang,
        "search": entity_name,
        "limit": limit
    }

    headers = {
        "User-Agent": USER_AGENT
    }

    try:
        response = requests.get(WIKIDATA_API, params=params, headers=headers, timeout=15)
        response.raise_for_status()
        data = response.json()

        results = data.get("search", [])
        if not results:
            return None, 0.0, None, None, None

        best_candidate = None
        best_confidence = -1.0

        for candidate in results:
            qid = candidate.get("id")
            label = candidate.get("label", "")
            desc = candidate.get("description", "")

            label_similarity = similarity(entity_name, label)

            confidence = label_similarity
            if desc:
                confidence += 0.05

            confidence = min(confidence, 1.0)

            if confidence > best_confidence:
                best_confidence = confidence
                best_candidate = candidate

        if best_candidate is None:
            return None, 0.0, None, None, None

        qid = best_candidate.get("id")
        label = best_candidate.get("label", "")
        desc = best_candidate.get("description", "")
        uri = f"http://www.wikidata.org/entity/{qid}"

        return uri, round(best_confidence, 3), desc, qid, label

    except Exception as e:
        print(f"Error for {entity_name}: {e}")
        return None, 0.0, None, None, None

**Recuperated entities:**

In [ ]:
df_entities = pd.read_csv(EXTRACTED_ENTITIES_FILE)

entities = (
    df_entities["entity"]
    .dropna()
    .astype(str)
    .str.strip()
    .replace("", pd.NA)
    .dropna()
    .unique()
    .tolist()
)

print(f"Number of entities in {EXTRACTED_ENTITIES_FILE}: {len(entities)}")

**Mapping table filling:**

As mentionned above, here we regroup the informations ofr each entity (with WIKIDATA informations too) in the mapping table.

In [ ]:
mapping_rows = []

for entity in entities:
    external_uri, confidence, description, wikidata_id, wikidata_label = search_wikidata(entity)

    mapping_rows.append({
        "private_entity": entity,
        "external_uri": external_uri,
        "confidence": confidence,
        "description": description,
        "wikidata_id": wikidata_id,
        "wikidata_label": wikidata_label
    })

    print(f"{entity} -> {external_uri} | confidence={confidence}")
    time.sleep(0.2)

mapping_df = pd.DataFrame(mapping_rows)
mapping_df.to_csv(MAPPING_OUTPUT, index=False, encoding="utf-8")

print(f"\nLength of mapping table: {len(mapping_df)}")
print(mapping_df.head(10))

**Alignement:**

Then, we will create "owl:sameAs" links (for the entities for which we have found an equivalent in wikidata) so that we can have the form of ":MarieCurie owl:sameAs wd:Q7186".

In [ ]:
found_matches = mapping_df[mapping_df["external_uri"].notna()].copy()
print(f"\nNumber of entities found in Wikidata: {len(found_matches)}")

alignment_graph = Graph()

for _, row in found_matches.iterrows():
    local_uri = URIRef(make_local_uri(row["private_entity"]))
    wiki_uri = URIRef(row["external_uri"])
    alignment_graph.add((local_uri, OWL_SAMEAS, wiki_uri))

alignment_graph.serialize(ALIGNMENT_OUTPUT, format="nt", encoding="utf-8")
print(f"Alignment triples number: {len(alignment_graph)}")

In addition, we will also separate the entities found in WIKIDATA and the ones that we didn't found:

In [ ]:
found_entities = mapping_df[mapping_df["external_uri"].notna()].copy()
found_entities.to_csv(FOUND_OUTPUT, index=False, encoding="utf-8")
print(f"Entities found in Wikidata: {len(found_entities)}")

not_found_entities = mapping_df[mapping_df["external_uri"].isna()].copy()
not_found_entities.to_csv(NOT_FOUND_OUTPUT, index=False, encoding="utf-8")
print(f"Entities not found in Wikidata: {len(not_found_entities)}")

**Enrichement of the RDF Graph with the entities that we did not found:**

In [ ]:
kb_graph = Graph()

try:
    kb_graph.parse(KNOWLEDGE_BASE_INPUT, format="nt")
    print(f"\nKnowledge base loaded: {KNOWLEDGE_BASE_INPUT}")
except Exception as e:
    print(f"\nError in charging {KNOWLEDGE_BASE_INPUT}: {e}")
    kb_graph = Graph()

EX = Namespace(LOCAL_ENTITY_BASE)
REL = Namespace(LOCAL_RELATION_BASE)

kb_graph.bind("ex", EX)
kb_graph.bind("rel", REL)

not_found = mapping_df[mapping_df["external_uri"].isna()].copy()

print(f"\nAdd of the {len(not_found)} entities that we did not find in Wikidata in the RDF\n")

for _, row in not_found.iterrows():
    entity_label = str(row["private_entity"]).strip()
    entity_uri = URIRef(make_local_uri(entity_label))

    kb_graph.add((entity_uri, RDF.type, RDFS.Resource))
    kb_graph.add((entity_uri, RDFS.label, Literal(entity_label, lang="en")))
    kb_graph.add((entity_uri, RDFS.comment, Literal("Local entity not found in Wikidata", lang="en")))

    if pd.notna(row.get("description")) and str(row["description"]).strip():
        kb_graph.add((entity_uri, RDFS.comment, Literal(str(row["description"]), lang="en")))

kb_graph.serialize(ENRICHED_KB_OUTPUT, format="nt", encoding="utf-8")

print(f"Number of triples in the graph: {len(kb_graph)}")

# Phase 5 : Predicate Alignment Using SPARQL

Now, that we have aligned the entities, we need to do the same for the properties. In order to do that, we will extract the properties of the RDF graph and then for those who have a subject and an object that are aligned with WIKIDATA, we will extract using a SPARQL query the candidate properties from WIKIDATA. The final alignment is then obtained through a manual validation step.

**Confirguration:**

We will keep the same configuration as before. We will add the files "property_alignment_output" that will hold the aligned properties, the "predicate_candidates_putput" that will containes the predicates obtained from WIKIDATA and finally the "predicate_validation_file" to check the correspondance with our predicate. Moreover, we will add the SPARQL URL for querying WIKIDATA.

In [ ]:
from rdflib.namespace import OWL

ENTITY_ALIGNMENT_FILE = ALIGNMENT_OUTPUT
KB_INPUT = ENRICHED_KB_OUTPUT

WIKIDATA_SPARQL = "https://query.wikidata.org/sparql"

PREDICATE_CANDIDATES_OUTPUT = "predicate_candidates.csv"
PREDICATE_VALIDATION_FILE = "predicate_validation_template.csv"
PROPERTY_ALIGNMENT_OUTPUT = "property_alignment.nt"

Then, we create a display function nammed "shorten_url" that will allows us to simplify the writting. We will have "ex:MarieCurie" instead of a long URI like "http://example.org/entity/MarieCurie".

In [ ]:
def shorten_uri(uri):
    uri = str(uri)
    if uri.startswith(LOCAL_ENTITY_BASE):
        return "ex:" + uri.replace(LOCAL_ENTITY_BASE, "")
    if uri.startswith(LOCAL_RELATION_BASE):
        return "rel:" + uri.replace(LOCAL_RELATION_BASE, "")
    if uri.startswith("http://www.wikidata.org/entity/"):
        return "wd:" + uri.replace("http://www.wikidata.org/entity/", "")
    if uri.startswith("http://www.wikidata.org/prop/direct/"):
        return "wdt:" + uri.replace("http://www.wikidata.org/prop/direct/", "")
    return uri

Then, we need to charge the alignement file of the entities.

We will create a dictionnary "entity_to_wikidata" that will contain the original entity and its WIKIDATA entity that match with it. 

In [ ]:
alignment_graph = Graph()
alignment_graph.parse(ENTITY_ALIGNMENT_FILE, format="nt")

entity_to_wikidata = {}

for s, p, o in alignment_graph:
    if p == OWL.sameAs:
        entity_to_wikidata[str(s)] = str(o)

print(f"Number of aligned entities: {len(entity_to_wikidata)}")

for i, (local_uri, wd_uri) in enumerate(entity_to_wikidata.items()):
    print(shorten_uri(local_uri), "its entity wikidata:", shorten_uri(wd_uri))
    if i >= 4:
        break

Once we have charged the aligned entities, we need to charge our KB graph that contains our triplets.

In [ ]:
kb_graph = Graph()
kb_graph.parse(KB_INPUT, format="nt")

print(f"Number of triples in the KB: {len(kb_graph)}")

Now, we can extract the eligible triples. In other words, we extract the triplets that have aligned subject and object. If the subject and the object is not in the aligned entities file (with an equivalent in WIKIDATA), we do not extract them.

In [ ]:
import pandas as pd
import re

mapping_df = pd.read_csv("entity_linking_mapping.csv")
rel_df = pd.read_csv("relations.csv")

name_to_qid = {}
for _, row in mapping_df.iterrows():
    if pd.notna(row['wikidata_id']):
        name = str(row['private_entity']).lower().strip()
        qid = row['wikidata_id']
        name_to_qid[name] = qid

        parts = name.split()
        if len(parts) > 1:
            for part in parts:
                if len(part) > 3 and part not in name_to_qid:
                    name_to_qid[part] = qid

candidate_triples = []

for _, row in rel_df.iterrows():
    s_raw = str(row['subject']).lower().strip()
    o_raw = str(row['object']).lower().strip()
    
    s_qid = next((qid for name, qid in name_to_qid.items() if name in s_raw or s_raw in name), None)
    o_qid = next((qid for name, qid in name_to_qid.items() if name in o_raw or o_raw in name), None)

    if s_qid and o_qid:
        candidate_triples.append({
            "subject_local": row['subject'],
            "predicate_local": row['predicate'],
            "object_local": row['object'],
            "subject_wikidata": f"http://www.wikidata.org/entity/{s_qid}",
            "object_wikidata": f"http://www.wikidata.org/entity/{o_qid}"
        })

candidate_triples_df = pd.DataFrame(candidate_triples).drop_duplicates()
print(f"Number of eligible triples: {len(candidate_triples_df)}")

In order to avoid the query overload on WIKIDATA, we will delete the duplicates. In that case, we will avoid querying wikidata on the same thing each time.

In [ ]:
candidate_triples_df = candidate_triples_df.drop_duplicates(
    subset=["predicate_local", "subject_wikidata", "object_wikidata"]
).reset_index(drop=True)

print(f"After removing duplicates : {len(candidate_triples_df)}")
print(candidate_triples_df.head(10))

Now that we have extracted the eligibles properties (properties that have predicate and object with both an equivalent in WIKIDATA), we can query on WIIKIDATA to see their equivalent on it.

In [ ]:
import requests
import time
import pandas as pd
from difflib import SequenceMatcher

def similarity(a, b):
    return SequenceMatcher(None, str(a).lower(), str(b).lower()).ratio()

def get_all_subject_claims(s_qid):
    query = f"""
    SELECT ?propertyLabel ?valueLabel WHERE {{
      wd:{s_qid} ?p ?statement .
      ?property wikibase:claim ?p .
      ?statement ?ps ?value .
      ?property wikibase:statementProperty ?ps .
      SERVICE wikibase:label {{ bd:serviceParam wikibase:language "en". }}
    }}
    """
    try:
        r = requests.get("https://query.wikidata.org/sparql", params={'query': query, 'format': 'json'}, timeout=15)
        return [(row['propertyLabel']['value'], row['valueLabel']['value']) for row in r.json()['results']['bindings']]
    except:
        return []

final_results = []
print(f"Scan of the {len(candidate_triples_df)} subjects...")

for _, row in candidate_triples_df.iterrows():
    s_qid = row['subject_wikidata'].split('/')[-1]
    o_text = str(row['object_local']).lower().strip()
    p_local = str(row['predicate_local']).lower().strip()
    
    claims = get_all_subject_claims(s_qid)
    
    for p_label, v_label in claims:
        score_obj = similarity(o_text, v_label)
        score_pred = similarity(p_local, p_label)
        
        if score_obj > 0.7 or score_pred > 0.7:
            final_results.append({
                "local_subject": row['subject_local'],
                "local_predicate": row['predicate_local'],
                "local_object": row['object_local'],
                "wikidata_property": p_label,
                "wikidata_value": v_label,
                "score": max(score_obj, score_pred)
            })
    time.sleep(0.3)

if final_results:
    final_df = pd.DataFrame(final_results).sort_values(by="score", ascending=False).drop_duplicates()
    final_df.to_csv("final_alignment_fix.csv", index=False)
    print(f"{len(final_df)} correspondances found")
    print(final_df.head(10))
else:
    print("No correspondences found")

In [ ]:
candidate_triples_df.to_csv("candidate_triples_for_alignment.csv", index=False)

As for the entities, we will extract the candidates of properties because each time we can have few values available on WIKIDATA. Each time, we use the similarity function (created in step 4), in order to compare the property that is in our data and the candidate property available on WIKIDATA.

In [ ]:
import requests

def qid_from_uri(uri: str) -> str:
    return str(uri).rstrip("/").split("/")[-1]

def get_wikidata_properties_between(subject_uri: str, object_uri: str):
    s_qid = qid_from_uri(subject_uri)
    o_qid = qid_from_uri(object_uri)

    query = f"""
    SELECT DISTINCT ?property ?propertyLabel WHERE {{
      wd:{s_qid} ?p wd:{o_qid} .
      ?property wikibase:directClaim ?p .
      SERVICE wikibase:label {{ bd:serviceParam wikibase:language "en". }}
    }}
    """

    try:
        r = requests.get(
            WIKIDATA_SPARQL,
            params={"query": query, "format": "json"},
            headers={"User-Agent": USER_AGENT},
            timeout=30
        )
        r.raise_for_status()
        data = r.json()["results"]["bindings"]

        return [
            (row["property"]["value"], row["propertyLabel"]["value"])
            for row in data
        ]
    except Exception as e:
        print(f"SPARQL error between {s_qid} and {o_qid}: {e}")
        return []

In [ ]:
candidate_rows = []

print(f"Analyzing {len(candidate_triples_df)} eligible triplets")

for _, row in candidate_triples_df.iterrows():
    s_wd = row["subject_wikidata"]
    o_wd = row["object_wikidata"]
    
    # normal direction
    props = get_wikidata_properties_between(s_wd, o_wd)
    # inverse direction
    if not props:
        props = get_wikidata_properties_between(o_wd, s_wd)
        
    for prop_uri, prop_label in props:
        local_pred_name = shorten_uri(row["predicate_local"]).split(":")[-1]
        score = similarity(local_pred_name, prop_label)
        
        candidate_rows.append({
            "subject_local": row["subject_local"],
            "predicate_local": row["predicate_local"],
            "object_local": row["object_local"],
            "subject_wikidata": s_wd,
            "object_wikidata": o_wd,
            "candidate_property_uri": prop_uri,
            "candidate_property_label": prop_label,
            "text_similarity": round(score, 3)
        })
    time.sleep(0.1)

predicate_candidates_df = pd.DataFrame(candidate_rows)
predicate_candidates_df = predicate_candidates_df.drop_duplicates(subset=["predicate_local", "candidate_property_uri"])

Now that we have a file with all the candidates predicates, we will select the correct ones by putting "yes" in the last column. Then, once this is done, the code below will allow us to keep only the predicating corresponding to the lines with a "yes".

In [ ]:
validated_df = predicate_candidates_df.copy()

validated_df["validated_relation"] = ""
validated_df["is_selected"] = ""

validated_df.to_csv(PREDICATE_VALIDATION_FILE, index=False, encoding="utf-8")

print(validated_df.head())

In [ ]:
validated_df = pd.read_csv(PREDICATE_VALIDATION_FILE)

validated_df = validated_df[
    (validated_df["is_selected"].astype(str).str.lower() == "yes") &
    (validated_df["validated_relation"].notna()) &
    (validated_df["validated_relation"].astype(str).str.strip() != "")
].copy()

print(f"Number of validated alignments: {len(validated_df)}")

print("\nPreview:")
print(validated_df[[
    "predicate_local",
    "candidate_property_uri",
    "candidate_property_label",
    "validated_relation"
]].head())

**Final RDF: transformation of the csv file into an RDF**

Finally, we will transform the csv file of the predicates into an RDF Graph.

In [ ]:
property_alignment_graph = Graph()

for _, row in validated_df.iterrows():
    local_pred = URIRef(row["predicate_local"])
    wd_prop = URIRef(row["candidate_property_uri"])
    relation_type = str(row["validated_relation"]).strip()

    if relation_type == "equivalentProperty":
        property_alignment_graph.add((local_pred, OWL.equivalentProperty, wd_prop))
    elif relation_type == "subPropertyOf":
        property_alignment_graph.add((local_pred, RDFS.subPropertyOf, wd_prop))

property_alignment_graph.serialize(PROPERTY_ALIGNMENT_OUTPUT, format="nt", encoding="utf-8")

print(f"Generated file: {PROPERTY_ALIGNMENT_OUTPUT}")
print(f"Number of alignment triplets: {len(property_alignment_graph)}")

# Phase 6 : KB Expansion via SPARQ

In the previous step, we searched for the eligible predicates (the one that their aligned subject and object are available in the entities found in wikidata) on WIKIDATA. Therefore, we succeded to find 25 triplets. Now, we will use KB expansion via SPARQL in order to enriche our database using data contained in WIKIDATA. For that, we will use the One Hop expansion and the Two Hop expansion. The principle is that the One Hop expansion is a principle based on a query that will allows us to find and extract the neighbors of each of ours entities available on wikidata. In that way we will be able to have more data about our subject. Concerning the Two Hop expansion, we will use the relations extracted in the One Hop expansion in order to generate their respective relations. Once again, we will have a more enriched graph as we will have the relations of the relations in addition of the original relations obtained from the articles and their wikidata equivalent.

**Configuration needed for the KB Expansion:**

In fact, we fix the WIKIDATA API link in the variable "WIKIDATA_SPARQL" and we also we fix the agent for scrawling and the namespaces in order to simplify the URIs.

In [ ]:
import requests
import time
import pandas as pd
from rdflib import Graph, URIRef, Literal, Namespace
from rdflib.namespace import OWL

WIKIDATA_SPARQL = "https://query.wikidata.org/sparql"

HEADERS = {
    "Accept": "application/sparql-results+json",
    "User-Agent": USER_AGENT
}

EX = Namespace(LOCAL_ENTITY_BASE)
REL = Namespace(LOCAL_RELATION_BASE)
WD = Namespace("http://www.wikidata.org/entity/")
WDT = Namespace("http://www.wikidata.org/prop/direct/")

Another important point of the configuration is the parameters of the SPARQL queries. We will fix the limit of the one hop, the limit of the two hop, the max core entities as well as the confidence threshold. Indeed, the One Hop can have thousands and thousands of relations, so we will limitate it to 200 as otherwise we have the risk to explose our graph. It's the same for the Two Hop that can generate even more relations as the neighbor of the neighbor can really become a very large graph. This is why we fixed a limit of 40. Moreover, we also fixed a limit of 80 as it will reduce the code cost and avoid the reduce of the efficiency.

Concerning the request sleep, we will fix it as 0.15 so that we can avoid the spaming of queries in WIKIDATA.

Finally, we also fixed a confidence threshold on zero as we wants to keep all the entities in order to improve the number of our results.

In [ ]:
MAX_CORE_ENTITIES = 300
ONE_HOP_LIMIT = 200
TWO_HOP_ENTITY_LIMIT = 15
TWO_HOP_LIMIT = 40
REQUEST_SLEEP = 0.5
CONFIDENCE_THRESHOLD = 0.0

**Utility functions:**

In [ ]:
def qid_from_uri(uri: str) -> str:
    return str(uri).split("/")[-1]

def sparql_select(query: str):
    try:
        response = requests.get(
            WIKIDATA_SPARQL,
            params={"query": query, "format": "json"},
            headers=HEADERS,
            timeout=60
        )
        response.raise_for_status()
        return response.json()["results"]["bindings"]
    except Exception as e:
        print("SPARQL error:", e)
        return []

def make_node(binding, var):
    if var not in binding:
        return None
    
    val = binding[var]
    
    if val["type"] == "uri":
        return URIRef(val["value"])
    elif val["type"] == "literal":
        return Literal(val["value"])
    return None

**Extraction of the aligned entities:**

Then, we will charge the aligned entities so that we can then enriche them.

In [ ]:
alignment_graph = Graph()
alignment_graph.parse(ALIGNMENT_OUTPUT, format="nt")

entity_to_wikidata = {}

for s, p, o in alignment_graph:
    if p == OWL.sameAs:
        entity_to_wikidata[str(s)] = str(o)

print("Aligned entities:", len(entity_to_wikidata))

Once we have extract the aligned entities, we will extract our KB Graph that we wants to enrich. Here, in the code below we will plot the number of triplet before expansion.

In [ ]:
kb_graph = Graph()
kb_graph.parse(ENRICHED_KB_OUTPUT, format="nt")

print("Triples before expansion:", len(kb_graph))

**Definition of the One hop and two hop functions:**

After having charged the graph and the entities, we will implement the functions "get_one_hop" and "get_two_hops" that will allow us to make the one hop and two hop queries. 

In [ ]:
def get_one_hop(qid):
    query = f"""
    SELECT ?p ?o WHERE {{
      wd:{qid} ?p ?o .
    }}
    LIMIT {ONE_HOP_LIMIT}
    """
    return sparql_select(query)

def get_two_hop(qid):
    query = f"""
    SELECT ?p ?o WHERE {{
      wd:{qid} ?p ?o .
    }}
    LIMIT {TWO_HOP_LIMIT}
    """
    return sparql_select(query)

## One hop expansion

In [ ]:
added_one_hop = 0

for i, (local_uri, wd_uri) in enumerate(list(entity_to_wikidata.items())[:MAX_CORE_ENTITIES]):
    
    qid = qid_from_uri(wd_uri)
    subject = URIRef(local_uri)
    
    print(f"[{i+1}] Expanding {qid}")
    
    results = get_one_hop(qid)
    
    for r in results:
        p = make_node(r, "p")
        o = make_node(r, "o")
        
        if p and o:
            triple = (subject, p, o)
            
            if triple not in kb_graph:
                kb_graph.add(triple)
                added_one_hop += 1
    
    time.sleep(REQUEST_SLEEP)

print("1-hop added:", added_one_hop)

## Two Hop expansion

In [ ]:
added_two_hop = 0

for i, (local_uri, wd_uri) in enumerate(list(entity_to_wikidata.items())[:MAX_CORE_ENTITIES]):
    
    qid = qid_from_uri(wd_uri)
    subject = URIRef(local_uri)
    
    results = get_one_hop(qid)
    
    entity_objects = []
    
    for r in results:
        o = make_node(r, "o")
        if isinstance(o, URIRef) and "wikidata.org/entity" in str(o):
            entity_objects.append(o)
    
    entity_objects = entity_objects[:TWO_HOP_ENTITY_LIMIT]
    
    for obj in entity_objects:
        obj_qid = qid_from_uri(obj)
        
        results_2 = get_two_hop(obj_qid)
        
        for r2 in results_2:
            p2 = make_node(r2, "p")
            o2 = make_node(r2, "o")
            
            if p2 and o2:
                triple = (obj, p2, o2)
                
                if triple not in kb_graph:
                    kb_graph.add(triple)
                    added_two_hop += 1
        
        time.sleep(REQUEST_SLEEP)

print("2-hop added:", added_two_hop)

**Save of the KB graph that we have expanded into a "nt" format:**

In [ ]:
kb_graph.serialize("knowledge_base_expanded.nt", format="nt")

print("Saved expanded KB")

**Final results:**

Now that we have expanded our KB graph, we will check if we have enough triplets, entities and relations after expansion.

In [ ]:
entities = set()
relations = set()

for s, p, o in kb_graph:
    entities.add(s)
    relations.add(p)
    if isinstance(o, URIRef):
        entities.add(o)

print("Final triples:", len(kb_graph))
print("Entities:", len(entities))
print("Relations:", len(relations))

we are in the right range for the number of triples and entities. However, we have 2046 relations which exceed a lot the range of 50-200. In order to reduce the number of relations, we will extract the most frequent relations and keep the 150 most fequent ones. 

**Count of the frequence of each relation:**

In [ ]:
from collections import Counter

relation_counts = Counter()

for s, p, o in kb_graph:
    relation_counts[p] += 1

print("Total relations:", len(relation_counts))

**Sort of the relations from the most frequent relations to the less frequent ones:**

In [ ]:
sorted_relations = sorted(
    relation_counts.items(),
    key=lambda x: x[1],
    reverse=True
)

for rel, count in sorted_relations[:10]:
    print(rel, count)

**Keep of the 150 most frequent ones:**

In [ ]:
TOP_K = 150

top_relations = set([rel for rel, _ in sorted_relations[:TOP_K]])

print("Relations kept:", len(top_relations))

**Add changes to the graph:**

In [ ]:
from rdflib import Graph

filtered_graph = Graph()

for s, p, o in kb_graph:
    if p in top_relations:
        filtered_graph.add((s, p, o))

In [ ]:
entities = set()
relations = set()

for s, p, o in filtered_graph:
    entities.add(s)
    relations.add(p)
    if isinstance(o, URIRef):
        entities.add(o)

print("Filtered triples:", len(filtered_graph))
print("Entities:", len(entities))
print("Relations:", len(relations))

In [ ]:
filtered_graph.serialize("knowledge_base_filtered.nt", format="nt")

print("Saved filtered KB")

# Phase 7 : Knowledge reasoning with SWRL

In this step, we will load the family.owl file. It's a file composed of an ontology with normally classes et individuals. The goal will be to apply SWRL a rule that state "a person who is older than 60 years old is an 'oldPerson'" ('oldPerson' is a class).

In order to do that, we need to first charge the "family.owl" file and for that we need to dowload the OWLReady2 in python. It will allow us to charge the ontology contained in the 'family.owl" file.

**Installation of OWLReady2:**

In [ ]:
!pip install owlready2

**Charge of the ontology:**

In [ ]:
from owlready2 import *
onto = get_ontology("family.owl").load()
print("Ontology loaded:", onto)

Usually in a OWL file we have an ontology with classes and individuals. Let's check the content of this file in terms of classes and individuals number.

In [ ]:
print("Classes:")
for cls in onto.classes():
    print("-", cls)

print("\nIndividuals:")
for ind in onto.individuals():
    print("-", ind)

We have well the class "Person". However, we do not have any individual, this is why we will add some individuals so that we can be able to test our query.

In [ ]:
if len(list(onto.individuals())) == 0:
    with onto:
        john = onto.Person("John")
        john.age = 65
        john.name = "John"

        mary = onto.Person("Mary")
        mary.age = 45
        mary.name = "Mary"

        paul = onto.Person("Paul")
        paul.age = 72
        paul.name = "Paul"

    print("Test individuals created.")
else:
    print("Individuals already exist in the ontology.")

In [ ]:
print("Individuals and their ages:")
for ind in onto.individuals():
    print(ind, "| age =", getattr(ind, "age", None))

Then, we will also check the list of the data and object properties:

In [ ]:
print("Object properties:")
for prop in onto.object_properties():
    print("-", prop)

print("\nData properties:")
for prop in onto.data_properties():
    print("-", prop)

In [ ]:
with onto:
    class oldPerson(onto.Person):
        pass

print(onto.oldPerson)

In [ ]:
print(hasattr(onto, "oldPerson"))
print(onto.oldPerson)

Now that we have check he presence of the "Person" class, the object property "hasAge", we can add the SWRL rule.

In [ ]:
with onto:
    rule = Imp()
    rule.set_as_rule("""
        Person(?p), age(?p, ?a), greaterThan(?a, 60) -> oldPerson(?p)
    """)

**Now, we can the display the SWRL rule:**

In [ ]:
print("SWRL rule:")
print(rule)

Once the query is set up, we need to launch the reasoner so that we can apply the rule and therefore the reasoner will add the consequences of this rule to the ontology.

In [ ]:
from owlready2 import *
sync_reasoner_pellet(infer_property_values=True, infer_data_property_values=True)

Finally, let's check if the rule has been added successefully and the reasoner has done the logical reasoning by plotting the individuals. Normally, we should have individuals that have 60 years old and more.

In [ ]:
print("Individuals inferred as oldPerson:")
for ind in onto.oldPerson.instances():
    print("-", ind)

According to the results above, we have well only individuals with an age that is greater than 60 years old.

# Phase 8 : Knowledge Graph Embedding

## 1 — Data Preparation

### 1.1 Input

Before applying any embedding model, we will need to charge our RDF graph made in the last step. Moreover, it will also be necessary to clean and prepare it for applying the embedding model.

In [ ]:
from rdflib import Graph

g = Graph()
g.parse("knowledge_base_filtered.nt", format="nt")

print("Number of triples:", len(g))

### 1.2 Cleaning for Embedding

Now that we have extracted our graph, we will simplify the URIs so that we can avoid the long ones. Indeed, we wil recuperate the last member of the URI.

In [ ]:
def shorten_uri(uri):
    uri = str(uri)
    
    if uri.startswith("http://example.org/entity/"):
        return uri.split("/")[-1]
    
    if uri.startswith("http://www.wikidata.org/entity/"):
        return uri.split("/")[-1]
    
    if uri.startswith("http://www.wikidata.org/prop/direct/"):
        return uri.split("/")[-1]
    
    return uri

Then, we will save the triplets in a ".txt" format:

In [ ]:
with open("triples_raw.txt", "w", encoding="utf-8") as f:
    for s, p, o in g:
        s_clean = shorten_uri(s)
        p_clean = shorten_uri(p)
        o_clean = shorten_uri(o)
        
        f.write(f"{s_clean}\t{p_clean}\t{o_clean}\n")

Following that, we will also need to do some supplementary cleaning before applying any embedding. This is why, we will also check if there are duplicates and we will delete them. 

In [ ]:
triples = set()

with open("triples_raw.txt", "r", encoding="utf-8") as f:
    for line in f:
        triples.add(line.strip())

print("Unique triples:", len(triples))

As we already mentionned before, we need to filter the incoherent URIs. In other words, we need to have QID that does not have problems in their form. 

In [ ]:
import re

def is_valid_entity(x):
    if x.startswith("Q") and re.match(r"^Q\d+$", x):
        return True

    if re.match(r"^[a-zA-Z0-9_]+$", x):
        return True
    return False

Finally, we will stock all the cleaned triples in an array. For each triplets, we will use the function "is_valid_entity" in order to delete the invalid entities. 

In [ ]:
cleaned_triples = []
for triple in triples:
    parts = triple.split("\t")
    if len(parts) == 3:
        s, p, o = parts
        if is_valid_entity(s) and is_valid_entity(o):
            cleaned_triples.append((s, p, o))

print("Cleaned triples:", len(cleaned_triples))

### 1.3 Train / Validation / Test Split

Once we have extracted and cleaned our data, we can finally apply the embedding models. Concerning the implementation, we need to first split our data into training data, validation data and test data. We will fix the repartition as follow : 80% for the train, 10% for the validation and 10% for the test.

However, before applying this split, we need to shuffle our data in order to avoid the biases.

In [ ]:
import random
random.seed(42)
random.shuffle(cleaned_triples)

**Train / Validation / Test split:**

In [ ]:
n = len(cleaned_triples)

train = cleaned_triples[:int(0.8 * n)]
valid = cleaned_triples[int(0.8 * n):int(0.9 * n)]
test  = cleaned_triples[int(0.9 * n):]

Once we have done the split, we need to check if the test entities are well in the train data. 

In [ ]:
train_entities = set()
for s, p, o in train:
    train_entities.add(s)
    train_entities.add(o)

def is_valid(triple):
    s, p, o = triple
    return s in train_entities and o in train_entities

In [ ]:
test = [t for t in test if is_valid(t)]
valid = [t for t in valid if is_valid(t)]

Finally, we can save our files of testing data, valiation data and testing data:

In [ ]:
def save_triples(triples, filename):
    with open(filename, "w") as f:
        for s, p, o in triples:
            f.write(f"{s}\t{p}\t{o}\n")

save_triples(train, "train.txt")
save_triples(valid, "valid.txt")
save_triples(test, "test.txt")

In each file, we have this number of triples:

In [ ]:
print("Train:", len(train))
print("Valid:", len(valid))
print("Test:", len(test))

## 2 — Embedding Models

Now we can implement our embedding models. For that, we will choose two of the recommended models in the statement : TransE and ComplEx. Indeed, we can easily compare those two models.On the one hand, TransE is a transational model where the entities and relations are vectors. On the other hand, ComplEx is good as it handles asymetric relations.

However, in order to implement those two models, we need to have numerical values. For that, we will use the PyKEEN framework, as mentionned in the statement. This tool will allows us to convert our data and index it. In that case, each entity will have its own numerical ID. Moreover, PyKEEN will stock them in a dictionnary (mapping phase). 

In [ ]:
!pip install pykeen

In [ ]:
from pykeen.triples import TriplesFactory

training = TriplesFactory.from_path('train.txt')

validation = TriplesFactory.from_path(
    'valid.txt', 
    entity_to_id=training.entity_to_id, 
    relation_to_id=training.relation_to_id
)
testing = TriplesFactory.from_path(
    'test.txt', 
    entity_to_id=training.entity_to_id, 
    relation_to_id=training.relation_to_id
)

print(f"Number of unique entities indexed : {training.num_entities}")
print(f"Number of unique relations indexed : {training.num_relations}")

### TransE model

In order to implement our models, we need to fix parameters. However, as we want to compare them, we need to fix the same parameters for both of them. This is why we will stock their parameters in the dictionnary "common_kwargs". 

Concerning the parameters choice, we chose a dimension of "100" because it allows us to avoid over-fitting. Indeed, a high value of dimension can lead to over-fitting. Moreover, such a dimension, can allows us to capture a big variety of semantial caracteristics without being very complex. Moroever, for the learning rate, we chose the value "0.001" which is equivalent to the "Adam" value and its considered as one of the most efficient prameter, it's also the one used by default by PyKEEN. In addition, we fixed our batch size to 128, which means that each time we will take 128 triples and this will allows us to have a stable gradient. Finally, we will fix the number of epochs at 50 so that we can have a good balance between efficiency and time-consuming for our model training. The negative samples is fixed to "10" which means that we present 10 fake examples for only 1 true example. This will avoid the trivial answers of our model and make it more robust.

In [ ]:
DIMENSION = 100         
LEARNING_RATE = 0.001   
BATCH_SIZE = 128       
EPOCHS = 50             
NEG_SAMPLES = 10

In [ ]:
from pykeen.pipeline import pipeline  
import torch

result_transe = pipeline(
    training=training,
    validation=validation,
    testing=testing,
    model='TransE',
    model_kwargs=dict(embedding_dim=DIMENSION),
    optimizer_kwargs=dict(lr=LEARNING_RATE),
    training_kwargs=dict(
        num_epochs=EPOCHS,
        batch_size=BATCH_SIZE,
    ),
    negative_sampler='basic',
    negative_sampler_kwargs=dict(num_negs_per_pos=NEG_SAMPLES),
    random_seed=42,
    device='cuda' if torch.cuda.is_available() else 'cpu',
)
result_transe.save_to_directory('results_transe')

#### Evaluation of the  TransE model 

##### Global results of both head and tail:

In [ ]:
print("TransE - MRR:", result_transe.metric_results.get_metric("both.realistic.inverse_harmonic_mean_rank"))
print("TransE - Hits@1:", result_transe.metric_results.get_metric("both.realistic.hits_at_1"))
print("TransE - Hits@3:", result_transe.metric_results.get_metric("both.realistic.hits_at_3"))
print("TransE - Hits@10:", result_transe.metric_results.get_metric("both.realistic.hits_at_10"))

According to the results above, we see that the model has an average performance with an MRR of 20% which is a good result. This means that in average the correct entity appears in the fifth rank. 
Concerning the Hits@K scores, we see that the model tends to have difficulties in finding correct answers in the top 1. However, in the top 3 and top 10, TransE model succeeded to have a better number of correct entities with a score of 22% for the Hits@3 and 38% for the Hits@10. 


##### Results of head predictions

In [ ]:
print("Head - MRR:", result_transe.metric_results.get_metric("head.realistic.inverse_harmonic_mean_rank"))
print("Head - Hits@1:", result_transe.metric_results.get_metric("head.realistic.hits_at_1"))
print("Head - Hits@3:", result_transe.metric_results.get_metric("head.realistic.hits_at_3"))
print("Head - Hits@10:", result_transe.metric_results.get_metric("head.realistic.hits_at_10"))

According to the results of the head prediction, we see that the MRR is the same as for the general result (both head and tail). So, the model is consistent between his different tasks. Moreover, we see that the Hits@1 score is better for the head with a score of 12% comparing to 11% for the general. However, we have a little decrease for the Hits@10 which suggest that the model have less correct entities in the top 10 for the head. 

##### Results of tail predictions

In [ ]:
print("Tail - MRR:", result_transe.metric_results.get_metric("tail.realistic.inverse_harmonic_mean_rank"))
print("Tail - Hits@1:", result_transe.metric_results.get_metric("tail.realistic.hits_at_1"))
print("Tail - Hits@3:", result_transe.metric_results.get_metric("tail.realistic.hits_at_3"))
print("Tail - Hits@10:", result_transe.metric_results.get_metric("tail.realistic.hits_at_10"))

Here, for the tail, the MRR is a little bit above the head with an increase of 0.6. The model tends to classify better the right tails. However, the Hits@1 score descreased and is about 10% which means that the model lack of precision on the correct candidates in the top 1 for the tail. 
Finally, the model achieved good results for the top 10 as he succeeded to obtain a score of 41% of correct entities. 


### ComplEx model

As mentionned before, the parameters for both our embedding models are the same, therefore, we will keep the same parameters of TransE model.

In [ ]:
result_complex = pipeline(
    training=training,
    validation=validation,
    testing=testing,
    model='ComplEx',
    model_kwargs=dict(embedding_dim=DIMENSION),
    optimizer_kwargs=dict(lr=LEARNING_RATE),
    training_kwargs=dict(
        num_epochs=EPOCHS,
        batch_size=BATCH_SIZE,
    ),
    negative_sampler='basic',
    negative_sampler_kwargs=dict(num_negs_per_pos=NEG_SAMPLES),
    random_seed=42,
    device='cuda' if torch.cuda.is_available() else 'cpu',
)
result_complex.save_to_directory('results_complex')

#### Evaluation of the ComplEx model

##### Global results of both head and tail

In [ ]:
print("ComplEx - MRR:", result_complex.metric_results.get_metric("both.realistic.inverse_harmonic_mean_rank"))
print("ComplEx - Hits@1:", result_complex.metric_results.get_metric("both.realistic.hits_at_1"))
print("ComplEx - Hits@3:", result_complex.metric_results.get_metric("both.realistic.hits_at_3"))
print("ComplEx - Hits@10:", result_complex.metric_results.get_metric("both.realistic.hits_at_10"))

According to the results above, we clearly see that the ComplEx model have a lot of difficulties in catching the correct entities. The MRR is extremely low with a score of 0.01. This suggest that the average rank of the correct entity is about 100 which is not good at all. This means that the model does not learn correctly. 

In addition, the Hits@K scores (top 1, top 3 and top 10) also confirms this bad classification. Indeed, mostly all the scores are close to zero.


##### Results of head predictions

In [ ]:
print("Head - MRR:", result_complex.metric_results.get_metric("head.realistic.inverse_harmonic_mean_rank"))
print("Head - Hits@1:", result_complex.metric_results.get_metric("head.realistic.hits_at_1"))
print("Head - Hits@3:", result_complex.metric_results.get_metric("head.realistic.hits_at_3"))
print("Head - Hits@10:", result_complex.metric_results.get_metric("head.realistic.hits_at_10"))

According to the results of the head, we constate approximately the same results as for the general version (head and tail). This reinforce the idea that the ComplEx model is not efficient.

##### Results of tail predictions

In [ ]:
print("Tail - MRR:", result_complex.metric_results.get_metric("tail.realistic.inverse_harmonic_mean_rank"))
print("Tail - Hits@1:", result_complex.metric_results.get_metric("tail.realistic.hits_at_1"))
print("Tail - Hits@3:", result_complex.metric_results.get_metric("tail.realistic.hits_at_3"))
print("Tail - Hits@10:", result_complex.metric_results.get_metric("tail.realistic.hits_at_10"))

According to the results of the tail, we constate approximately the same results as for the general version (head and tail) as well as the head version. This reinforce more in more the idea that the ComplEx model is not efficient. We even have scores equal to zero!

### Interpretation and comparaison of the models

**Both (head and tail) comparaison:**

In [ ]:
import pandas as pd
comparison_df = pd.DataFrame([
    {
        "model": "TransE",
        "MRR": result_transe.metric_results.get_metric("both.realistic.inverse_harmonic_mean_rank"),
        "Hits@1": result_transe.metric_results.get_metric("both.realistic.hits_at_1"),
        "Hits@3": result_transe.metric_results.get_metric("both.realistic.hits_at_3"),
        "Hits@10": result_transe.metric_results.get_metric("both.realistic.hits_at_10"),
    },
    {
        "model": "ComplEx",
        "MRR": result_complex.metric_results.get_metric("both.realistic.inverse_harmonic_mean_rank"),
        "Hits@1": result_complex.metric_results.get_metric("both.realistic.hits_at_1"),
        "Hits@3": result_complex.metric_results.get_metric("both.realistic.hits_at_3"),
        "Hits@10": result_complex.metric_results.get_metric("both.realistic.hits_at_10"),
    }
])

comparison_df

**Head comparaison:**

In [ ]:
import pandas as pd

head_df = pd.DataFrame([
    {
        "Model": "TransE",
        "MRR": result_transe.metric_results.get_metric("head.realistic.inverse_harmonic_mean_rank"),
        "Hits@1": result_transe.metric_results.get_metric("head.realistic.hits_at_1"),
        "Hits@3": result_transe.metric_results.get_metric("head.realistic.hits_at_3"),
        "Hits@10": result_transe.metric_results.get_metric("head.realistic.hits_at_10"),
    },
    {
        "Model": "ComplEx",
        "MRR": result_complex.metric_results.get_metric("head.realistic.inverse_harmonic_mean_rank"),
        "Hits@1": result_complex.metric_results.get_metric("head.realistic.hits_at_1"),
        "Hits@3": result_complex.metric_results.get_metric("head.realistic.hits_at_3"),
        "Hits@10": result_complex.metric_results.get_metric("head.realistic.hits_at_10"),
    }
])

print("Head Prediction Results:")
head_df

**Tail comparaison:**

In [ ]:
tail_df = pd.DataFrame([
    {
        "Model": "TransE",
        "MRR": result_transe.metric_results.get_metric("tail.realistic.inverse_harmonic_mean_rank"),
        "Hits@1": result_transe.metric_results.get_metric("tail.realistic.hits_at_1"),
        "Hits@3": result_transe.metric_results.get_metric("tail.realistic.hits_at_3"),
        "Hits@10": result_transe.metric_results.get_metric("tail.realistic.hits_at_10"),
    },
    {
        "Model": "ComplEx",
        "MRR": result_complex.metric_results.get_metric("tail.realistic.inverse_harmonic_mean_rank"),
        "Hits@1": result_complex.metric_results.get_metric("tail.realistic.hits_at_1"),
        "Hits@3": result_complex.metric_results.get_metric("tail.realistic.hits_at_3"),
        "Hits@10": result_complex.metric_results.get_metric("tail.realistic.hits_at_10"),
    }
])

print("Tail Prediction Results:")
tail_df

**Interpreation of the results and comparaison:**

According to the results above, we clearly see that the TransE model is better than the ComplEx model. This result is valid for all the metrics : head and tail together, head alone and tail alone. Indeed, in all the metris, the MRR is about 0.20 for the TransE comparing to only 0.01 to ComplEx. This suggest that using the TransE model, the correct entity is usually the 5th position in the results. 

Moreover, concerning the Hits@K scores, which correspond to the pourcentage of time the right answer is in the top K, the TransE model outperform ComplEx model too. Indeed, for the TransE, the right entitiy is in top 1 in 10% (11% for both, 12% for head and 10% for tail) of the time whereas in the ComplEx model, it's present in only 0.004 % of the time. 

In addition, we also see the difference between the two models while we analyze the difference between the head and tail. Indeed, on the one hand, for the TransE model we have a good balance between the head and tail for the MRR with 0.2026 and 0.2069 respectively. However, it still have a very small gap for the Hits@10 that is higher in the tail with a pourcentage of 41% comparing to 36% in the head. this show us that the model predict better the objects than the subjects. 
On the other hand, for the ComplEx model, the head and tail are both very low (around 0.01) and therefore we does not have a useful learning.

Overall, we can say that the TransE model outperform the ComplEx model.This can be due to the fact that we kept the most frequent relations and therefore we simplified our graph. Moreover, TransE model is very good at catching simple structures and allows us to have good results here. On the contrast, ComplEx model has a lot of difficulties in performing and this can be explained by the size of our dataset or noise. If we cant to improve the result of the ComplEx mode, we could try to adapt the parameters(tuning).

**Save of the results in a csv file:**

In [ ]:
comparison_df.to_csv("embedding_model_comparison.csv", index=False)

result_transe.save_to_directory("results_transe")
result_complex.save_to_directory("results_complex")

head_df.to_csv("head_comparison.csv", index=False)
tail_df.to_csv("tail_comparison.csv", index=False)

## 3— KB Size Sensitivity

Now that we have see that TransE model is better than ComplEx model, we will divide our KB database into three dfferent subsamples : 20K triplets, 50k triplets and the full dataset. For each sample, we will aplly our TransE model that we have created just before.

**Extraction of the samples : 20k triplets of the dataset and 50k of the dataset:**

In [ ]:
import random

with open('train.txt', 'r') as f:
    all_triples = f.readlines()

random.seed(42)
random.shuffle(all_triples)

size_20k = min(20000, len(all_triples))
with open('train_20k.txt', 'w') as f:
    f.writelines(all_triples[:size_20k])

size_50k = min(50000, len(all_triples))
with open('train_50k.txt', 'w') as f:
    f.writelines(all_triples[:size_50k])

print(f"Files created : 20k ({size_20k} triplets) and 50k ({size_50k} triplets)")

In [ ]:
import pandas as pd
from pykeen.pipeline import pipeline

sizes = ['20k', '50k', 'full']
all_results = []

for size in sizes:
    print(f"\n>>> Entraînement TransE sur taille : {size}...")
    
    path = 'train.txt' if size == 'full' else f'train_{size}.txt'
    current_train = TriplesFactory.from_path(path)
    
    result = pipeline(
        training=current_train,
        testing=testing, 
        model='TransE',
        training_kwargs=dict(num_epochs=50, batch_size=128),
        device='cpu'
    )
    
    res_dict = {
        'KB_Size': size,
        'MRR': result.get_metric('mrr'),
        'Hits@1': result.get_metric('hits_at_1'),
        'Hits@3': result.get_metric('hits_at_3'),
        'Hits@10': result.get_metric('hits_at_10')
    }
    all_results.append(res_dict)

df_sensitivity = pd.DataFrame(all_results)
print("\nResults of sensitivity analysis:")
print(df_sensitivity.to_string(index=False))

**Interpretation of the results:**

According to the results above, the 20k sample dataset is better. Indeed, it achieved the better scores in all the metrics with 13% for the MRR, 6% for the Hits@1, 14% for the Hits@3 and finally 28% for the Hits@10, which is higher than the full dataset even if it still very close to the full dataset which indicate that the full dataset is also one of the best. 

However, we notice that the dataset with 20k samples is better than the one with 50k. It's sound quite unconsistent with the fact that the full dataset is better and therefore the more we have data, the more we have good result. I other word this disharmony, suggest that increasing the dataset size does not always lead to better performance. This can be explained by the noise of the data. Indeed, maybe the dataset with 50k samples introduces triplets that are less reliable and the relations less pertinent. 

Overall, those results show us that the quality and quantity of the data is very important for our knowledge graph embedding performance. So, one need to be careful while choosing the data filtering and sampling strategies.

## 4— Embedding Analysis

### 4.1 Nearest Neighbors

Now, we will try to recuperate the neighbors of each entity in the embedding space. In that way, we will be able to see if the TransE model (our better model) succedeed to regroup in the vector space, the entities that are really similar.

First, we will extract the matrix that contains the vectors of all the entities in 100 dimensions as well as the triples used in the training. 

Then, we iterate on each relation, and we identify for each relation its associate subject and we filter the very small categories (less than 3 entities) so that we can have a sgnificant statistical mean.

Following that, we calculate the centroids which are the points of gravity where the entities should convgerge to it. In addition, we compute the coherence using the Similar cosine. In fact, for each entity of a group, we calculate the angle between the vector and its centroid. If the score is equal to 1, then it means that we have a perfect match of the entity with its category. Finally, we measure the mean of those scores in order to have the coherence rate of the relation.

In [ ]:
import torch
import torch.nn.functional as F
import pandas as pd

entity_embeddings = result_transe.model.entity_representations[0](indices=None)

triples = training.mapped_triples
id_to_rel = {v: k for k, v in training.relation_to_id.items()}

def compute_semantic_rates():
    results = []
    
    for rel_id, rel_name in id_to_rel.items():
        subject_ids = triples[triples[:, 1] == rel_id][:, 0].unique()
        
        if len(subject_ids) >= 3:
            group_vectors = entity_embeddings[subject_ids]
            
            centroid = group_vectors.mean(dim=0, keepdim=True)
            
            similarities = F.cosine_similarity(group_vectors, centroid)
            
            coherence_rate = similarities.mean().item()
            
            results.append({
                'Category (Relation)': rel_name,
                'Number of Entities': len(subject_ids),
                'Coherence Rate (Cosine)': round(coherence_rate, 4)
            })

    return pd.DataFrame(results).sort_values(by='Coherence Rate (Cosine)', ascending=False)

df_rates = compute_semantic_rates()
print("Analysis of the semantic coherence")
print(df_rates.to_string(index=False))

global_rate = df_rates['Coherence Rate (Cosine)'].mean()
print(f"\nGlobal Coherence rate of the model: {global_rate:.4f}")

**Interpretation of the results:**

According to the results above, the TransE model succeded in creating solid geometrical clusters with a score of 67% for the relation "P5572" and 64% for the relation "P1476". This good result for those two entities can be explained by the fact that we have only 4 and 3 entities in those categories which facilitate their regroupment.

In addition, if we look at the evolution, we see that the more the number of entities increase, the more the coherence rate decrease. This is due tot he fact that the more we have diversity in the data, the more TransE will have difficulties to keep a big number of entities close to one centroid.

Finally, we see at the end, that the general relations such as the labels and versions have a very low coherent rate (12% and 7%) which is consitent with the fact that these relations do not define a precise semantic category. 

### 4.2 Clustering analysis

Now that we have seen the number of entities in each cluster (relation) and its coherence rate, we will do a visualization that plot those clusteers. In order to do that, we will use the tool t-SNE. This tool allows us to reduce the original dimension (100) into two dimension and therefore we will be able to plot the clusters.

(Note : I tried to apply the TSNE on all the entities but it crashed all my kernels and I had to re run all my project many times. There, I reduced the t-SNE to 1000 entities)

In [ ]:
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt
import torch
import numpy as np

n_total = len(training.entity_to_id)
n_samples = min(1000, n_total) 
indices = torch.randperm(n_total)[:n_samples]

embeddings_100d = result_transe.model.entity_representations[0](indices=indices).detach().cpu().numpy()

tsne = TSNE(n_components=2, perplexity=30, n_iter=1000, random_state=42, learning_rate='auto')
embeddings_2d = tsne.fit_transform(embeddings_100d)

id_to_entity = {v: k for k, v in training.entity_to_id.items()}

triples_np = training.mapped_triples.cpu().numpy()
entity_to_class = {row[0]: row[1] for row in triples_np}
colors = [entity_to_class.get(idx.item(), 0) for idx in indices]

plt.figure(figsize=(14, 10))
scatter = plt.scatter(embeddings_2d[:, 0], embeddings_2d[:, 1], 
                      c=colors, cmap='tab20', alpha=0.8, s=60, edgecolors='white', linewidth=0.5)

for i in range(min(15, n_samples)):
    plt.annotate(id_to_entity[indices[i].item()], (embeddings_2d[i, 0], embeddings_2d[i, 1]), 
                 fontsize=9, alpha=0.8, xytext=(5, 2), textcoords='offset points')

plt.title(f"Clustering Analysis : ({n_samples} entités)")
plt.colorbar(scatter, label='ID of the relation (classe)')
plt.grid(True, alpha=0.2)
plt.show()

**Do entities of the same class cluster together?**


According to the graph obtained above, we see that the relation ID are spread in micro-groups of identical colors that are very close together. For example, dark blue dots or the turquoise dots tend to "travel" together in small clusters. However, the regroupment is not clearly defined and perfect. This suggest that the entities in our dataset are multimodal and some entities can belong to few concepts.

**Does the embedding capture semantic structure?**

Yes the embedding captured the semantic structure. 

Indeed, as we can see on the graph, we have few places with different colors and we clearly see the difference in the location of the different entities. For example, we have the region "Idumea" and the language "arabe" that are located in precise locations in the graph. Thus, show that the model succeeded in creating logic (sense of the relations) with the geometry (the position in the space).

Moreover, we also see the micro-clusters on the graph. Indeed, here, the entities that belongs to the same class are not spread in the graph and they form micro-clusters. We can see that thanks to the mini regroupment of colors each time. This allows us to say that the model understood that those entities have properties and informations in common. However, we can note that we still have some entities that are overlapping each other (which is normal as one entity can have few roles).

Finally, we also see a specific hierarchical and contextual context. Indeed, in the middle we have the most connected entities and in the extremities of the graph we have the most rare concept.

Finally, we can say that our model succeeded in extracting the sense of our web pages.

### 4.3 Relations behavior

TransE model succeeded in capturing inverse relations and compositions. Indeed, this is mainly due to the fact that TransE is transational by nature. Here, this matched with our data well because inverse and composition relations are useful for informations extracted from WIKIPEDIA/WIKIDATA, which is our case in this project. Nevertheless, TransE model is not that well adapted for symetric relations as it would force the relation vector to zero. In that case, the ComplEx model is better for the asymmetric relations relation.

Finally, we can say that TransE model is adapted for inverse and composition relations and ComplEx is more adapted for the asymmetric relations and inverse relations.

## 5 — Critical Reflection

**Impact of predicate alignment quality:**

If we have some unaligned relations, we will create noise and therefore TransE model can create fake relations. This is probably what has been done on the entities that are at the boarders of the vosualization. Moreover, a unaligned predicated create a bad mean rank for everyone.

**Impact of noisy expansion:**

We have probably added noise by enriching informations in our base. Indeed, maybe it enriched our entity with similar words but maybe those words did not have the same meaning. This can create illogic schema in our graph and therefore this can have consequences on our vector embedding.

**Effect of ontology modeling choices:**

The ontology choice can have impacts on our study. Indeed, if we choose classes in a large way or in a very precise way, we can have embeddings that are located in the center. This is what se have seen in the graph of embedding visualization. Therefore, if the ontology is too much simple, it will make the entities converges in the center.

**Open-world assumption vs embedding assumptions:**

In our project, TensE model need to extrac t the triplets and for some triplets it's going to generate fake examples by creating triplets that are not in our database. However, aside of our dataset there are triplets that are correct, it's just that they are not in our database. Therefore the model is going to those vectors (subject and object) move away from each other even if they are words that match together. For example, if we have, (Beirut, capital of, Lebanon) if does not exist in our database and the model consider it as a negative example, it's going to move away Beirut from Lebanon whereas those two words are in reality close !

Finally, the previous lab (previous part of the project) had an important impact on our project and especially the scraping and parsing phase and wikidata alignement phase. Indeed, if we missed capital letter, our entities can be duplicated and therefore our model can create two different persons for the same entity. Moroever, if the triplets are not clean, we will not have a net t-SNE with structures and differents groups.


## 5 - Comparison between rule-based and embedding based reasoning

In this part we will compare the rule-based method and the embedding base reasoning.

For the SWRL method, we define a rule and then the reasoner make deductions. Here, in our project, we can propose this rule:

- BornIn(?p, ?v) ∧ isLocatedIn(?v, "Lebanon") → BornIn(?p, "Lebanon")

In contrast, in the embedding based reasoning method, we did not fix a rule. We only gave a lot of examples to our TensE model. The AI make the process alone and it goes from a person to a city and then to a country. It's simply the equivalent into vectors, such as :

$Vecteur(BornIn) + Vecteur(isLocatedIn) = Vecteur(BornInGlobal)$

So, in reality, with the embedding method, we achieve to the same conclusion thanks to the geometry. Finally, we can say that the embedding can find informations without having them in the database even if he is a little blurred. The embedding appears to be a simplified version and less strict version of the SWRL method.



# Phase 9 : Retrieval-Augmented Generation (RAG) with RDF/SPARQL and a Local Small LLM

In this final part of the project, we will create an LLM using the retriveal-augmented generation (RAG). This LLM should be able to request our RDF graph using SPARQL and give a result to the query.  

This part will be done in the ".py" file.